# 07. SQL Aggregations: Count, Sum, Avg & Stats: Beginner Guide

### 📝 SQL Execution Order:
```text
┌─ SQL Query Execution Order (Sequential Pipeline) ────────────────────────────┐
│ 1. FROM & JOIN (Load)    ➔ 2. WHERE (Filter)       ➔ 3. GROUP BY (Bucket)    │
│ ➔ 4. HAVING (Agg Filter) ➔ 5. SELECT (Pick Cols)   ➔ 6. DISTINCT (Dedup)     │
│ ➔ 7. ORDER BY (Sort)     ➔ 8. LIMIT / OFFSET (Page)                          │
└──────────────────────────────────────────────────────────────────────────────┘
```

---

### 📌 Overview & Architectural Context
Welcome to **07. SQL Aggregations: Count, Sum, Avg & Stats**. Aggregation functions execute vector reductions across multiple row tuples to produce consolidated statistical summaries. This notebook covers volume counting (`COUNT(*)`, `COUNT(col)`), unique cardinality counting (`COUNT(DISTINCT col)`), central tendency and dispersion metrics (`SUM`, `AVG`, `MIN`, `MAX`), aggregate null suppression mechanics, and conditional aggregations.

### 📚 Key Concepts Covered in this Notebook:
- [x] 🔹 Complete vs Attribute Counting: `COUNT(*)` vs `COUNT(column)`
- [x] 🔹 Cardinality Analysis: `COUNT(DISTINCT column)`
- [x] 🔹 Arithmetic Aggregations: `SUM()` & `AVG()`
- [x] 🔹 Boundary Value Extraction: `MIN()` & `MAX()`
- [x] 🔹 Aggregate Null Suppression Mechanics
- [x] 🔹 Conditional Aggregations: `CASE WHEN` Aggregation Pattern
- [x] 🔍 Scenario: Executive Payment Network Volume & Fraud Loss Summary








In [1]:
# Setup in-memory SQLite relational engine with Native SQL Studio Execution
import sqlite3
import pandas as pd
import os
from IPython import get_ipython
from IPython.core.magic import register_line_cell_magic

conn = sqlite3.connect(':memory:')

def load_table(name, path):
    if os.path.exists(path):
        df = pd.read_csv(path)
        df.to_sql(name, conn, index=False, if_exists='replace')

load_table('transactions', 'data/raw_transactions.csv' if os.path.exists('data/raw_transactions.csv') else '../data/raw_transactions.csv')
load_table('customers', 'data/customers.csv' if os.path.exists('data/customers.csv') else '../data/customers.csv')
load_table('merchants', 'data/merchants.csv' if os.path.exists('data/merchants.csv') else '../data/merchants.csv')
load_table('disputes', 'data/disputes.csv' if os.path.exists('data/disputes.csv') else '../data/disputes.csv')

def _execute_raw_sql(query):
    query = query.strip()
    if query.upper().startswith(('INSERT', 'UPDATE', 'DELETE', 'CREATE', 'DROP', 'ALTER', 'VACUUM', 'ANALYZE', 'BEGIN', 'COMMIT', 'ROLLBACK', 'SAVEPOINT')):
        cur = conn.cursor()
        cur.executescript(query)
        conn.commit()
        return "Query Executed Successfully."
    else:
        return pd.read_sql_query(query, conn)

# Register automatic raw SQL transformer & %%sql magic
ip = get_ipython()
if ip is not None:
    def raw_sql_transformer(lines):
        clean_text = ''.join(lines).strip()
        first_token = clean_text.split()[0].upper() if clean_text.split() else ''
        sql_keywords = {'SELECT', 'WITH', 'INSERT', 'UPDATE', 'DELETE', 'CREATE', 'DROP', 'ALTER', 'EXPLAIN', 'ANALYZE', 'VACUUM', 'BEGIN', 'COMMIT', 'ROLLBACK'}
        if first_token in sql_keywords:
            return [f'_execute_raw_sql("""{clean_text}""")']
        return lines
    
    if raw_sql_transformer not in ip.input_transformers_cleanup:
        ip.input_transformers_cleanup.append(raw_sql_transformer)

@register_line_cell_magic
def sql(line, cell=None):
    return _execute_raw_sql(cell if cell is not None else line)

print("SQL Studio Environment Active! You can now write and run pure SQL queries directly.")


SQL Studio Environment Active! You can now write and run pure SQL queries directly.


### 🔹 Volume Counting: `COUNT(*)` vs `COUNT(col)`
- **What it does:** `COUNT(*)` counts all candidate rows including nulls; `COUNT(column)` counts only non-null values in that specific attribute.
- **Syntax:** `SELECT COUNT(*), COUNT(column_name) FROM table_name`
- **Dataset Application & Code Demonstration:** Measures total transactions versus non-null transaction amounts in the dataset.


In [2]:
%%sql
SELECT 
    COUNT(*) AS total_records,
    COUNT(transaction_amount) AS valid_amount_records,
    (COUNT(*) - COUNT(transaction_amount)) AS missing_amount_count
FROM transactions;


,total_records,valid_amount_records,missing_amount_count
0,15000,14251,749


### 🔹 Cardinality Analysis: `COUNT(DISTINCT col)`
- **What it does:** Computes the exact number of unique, non-null values present in a column.
- **Syntax:** `SELECT COUNT(DISTINCT column_name) FROM table_name`
- **Dataset Application & Code Demonstration:** Counts unique customers and merchants active in the transaction log.


In [3]:
%%sql
SELECT 
    COUNT(DISTINCT customer_id) AS unique_customers,
    COUNT(DISTINCT merchant_id) AS unique_merchants,
    COUNT(DISTINCT card_type) AS unique_card_types
FROM transactions;


,unique_customers,unique_merchants,unique_card_types
0,1489,492,4


### 🔹 Mathematical Summaries: `SUM()`, `AVG()`, `MIN()`, `MAX()`
- **What it does:** Computes the total sum, arithmetic mean, minimum, and maximum across numeric column values.
- **Syntax:** `SELECT SUM(col), AVG(col), MIN(col), MAX(col) FROM table_name`
- **Dataset Application & Code Demonstration:** Calculates statistical volume benchmarks across all transactions.


In [4]:
%%sql
SELECT 
    ROUND(SUM(transaction_amount), 2) AS total_gross_volume,
    ROUND(AVG(transaction_amount), 2) AS average_ticket_size,
    MIN(transaction_amount) AS smallest_transaction,
    MAX(transaction_amount) AS largest_transaction
FROM transactions;


,total_gross_volume,average_ticket_size,smallest_transaction,largest_transaction
0,14326935.5,1005.33,5.02,1999.98


### 🔹 Conditional Aggregations: `CASE WHEN` Aggregation Pattern
- **What it does:** Embeds boolean conditional expressions inside aggregate functions to compute metric subtotals in a single table scan.
- **Syntax:** `SELECT SUM(CASE WHEN condition THEN amount ELSE 0 END) FROM table_name`
- **Dataset Application & Code Demonstration:** Computes total legitimate volume alongside total fraudulent volume in one pass.


In [5]:
%%sql
SELECT 
    ROUND(SUM(CASE WHEN is_fraud = 0 THEN transaction_amount ELSE 0 END), 2) AS legitimate_volume_usd,
    ROUND(SUM(CASE WHEN is_fraud = 1 THEN transaction_amount ELSE 0 END), 2) AS fraudulent_volume_usd,
    ROUND(100.0 * SUM(is_fraud) / COUNT(*), 2) AS fraud_rate_pct
FROM transactions;


,legitimate_volume_usd,fraudulent_volume_usd,fraud_rate_pct
0,11451183.65,2875751.85,10.77


## 💡 Real-World Practice & Scenarios
Practical scenarios and common data engineering questions explained with real examples.


### 🔍 Scenario: Q1: Aggregate Null Suppression Underflow & Averaging Gotchas
- **Objective:** Demonstrate how missing values skew `AVG(col)` if zeroes are encoded as `NULL` instead of `0`.
- **Approach:** Compare standard `AVG(col)` with `AVG(COALESCE(col, 0))`.


In [6]:
%%sql
SELECT 
    ROUND(AVG(transaction_amount), 2) AS avg_ignoring_nulls,
    ROUND(AVG(COALESCE(transaction_amount, 0)), 2) AS avg_coalescing_nulls_to_zero
FROM transactions;


,avg_ignoring_nulls,avg_coalescing_nulls_to_zero
0,1005.33,955.13
